<a href="https://colab.research.google.com/github/AditPradana36/m2f_playground/blob/main/Mask2Former_Comprehensive_Playground_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎭 Mask2Former — Universal Segmentation Playground

> **Comprehensive inference notebook** for every pretrained Mask2Former checkpoint on HuggingFace.
> Supports **Semantic**, **Panoptic**, and **Instance** segmentation across
> **ADE20K**, **Cityscapes**, **COCO**, and **Mapillary Vistas** datasets.

**Workflow:**
1. 🔧 **Cell 1** — Install dependencies *(run once)*
2. 🔗 **Cell 2** — Mount Google Drive
3. 🎛️ **Cell 3** — Select **Dataset, Task & Backbone** (via `@param` dropdowns)
4. 📂 **Cell 4** — Configure I/O paths
5. ⚙️  **Cell 5** — Set inference & visualisation options
6. 🤖 **Cell 6** — Load model *(auto-resolves HuggingFace checkpoint)*
7. 🚀 **Cell 7** — Run batch inference
8. 🖼️ **Cell 8** — Preview results
9. 📊 **Cell 9** — Export pixel stats (CSV / XLSX)
10. 📈 **Cell 10** — Class distribution chart

---
**References**

**Paper:** [Masked-attention Mask Transformer for Universal Image Segmentation (CVPR 2022)](https://arxiv.org/abs/2112.01527)  
**Model Zoo:** [github.com/facebookresearch/Mask2Former](https://github.com/facebookresearch/Mask2Former/blob/main/MODEL_ZOO.md)

---
This playground was developed by **[Mohammad Raditia Pradana](https://aditpradana36.github.io/)**.


*Please ensure the runtime is set to GPU. High-RAM runtime is recommended for optimal performance.

####Cell 1 — Install dependencies (run once)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 ── Install dependencies   (run once; restart runtime if prompted)
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip('transformers>=4.39.0', 'accelerate')
pip('torch', 'torchvision')          # already present in Colab; harmless
pip('Pillow', 'numpy', 'matplotlib', 'pandas', 'openpyxl', 'tqdm')

print('✅ All dependencies ready.')


#### Cell 2 — Mount Google Drive

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2 ── Mount Google Drive
# ═══════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')


####Cell 3 — Select Dataset, Task & Backbone (via @param dropdowns)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 ── 🎛️  MODEL SELECTION   ← use the Form panel (▶ button) ←
# ═══════════════════════════════════════════════════════════════════════

# @markdown ### 🗂️ Dataset
DATASET = 'ADE20K'  # @param ['ADE20K', 'Cityscapes', 'COCO', 'Mapillary Vistas']

# @markdown ### 🏷️ Segmentation Task
TASK = 'semantic'  # @param ['semantic', 'panoptic', 'instance']

# @markdown ### 🧠 Backbone
BACKBONE = 'Swin-L (IN21k)'  # @param ['ResNet-50', 'ResNet-101', 'Swin-T', 'Swin-S', 'Swin-B', 'Swin-B (IN21k)', 'Swin-L (IN21k)']

# ─── Model registry ─────────────────────────────────────────────────────────
# Key: (dataset_lower, task, backbone_key)  →  HuggingFace model id
# Source: https://github.com/facebookresearch/Mask2Former/blob/main/MODEL_ZOO.md
# All models are hosted under facebook/ on the HuggingFace Hub.

_REGISTRY = {
    # ── ADE20K ──────────────────────────────────────────────────────────────
    ('ade20k', 'semantic', 'resnet-50')      : 'facebook/mask2former-resnet50-ade-semantic',
    ('ade20k', 'semantic', 'resnet-101')     : 'facebook/mask2former-resnet101-ade-semantic',
    ('ade20k', 'semantic', 'swin-t')         : 'facebook/mask2former-swin-tiny-ade-semantic',
    ('ade20k', 'semantic', 'swin-s')         : 'facebook/mask2former-swin-small-ade-semantic',
    ('ade20k', 'semantic', 'swin-b')         : 'facebook/mask2former-swin-base-ade-semantic',
    ('ade20k', 'semantic', 'swin-b (in21k)') : 'facebook/mask2former-swin-base-in21k-ade-semantic',
    ('ade20k', 'semantic', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-ade-semantic',
    ('ade20k', 'panoptic', 'resnet-50')      : 'facebook/mask2former-resnet50-ade-panoptic',
    ('ade20k', 'panoptic', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-ade-panoptic',
    ('ade20k', 'instance', 'resnet-50')      : 'facebook/mask2former-resnet50-ade-instance',
    ('ade20k', 'instance', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-ade-instance',
    # ── Cityscapes ──────────────────────────────────────────────────────────
    ('cityscapes', 'semantic', 'resnet-50')      : 'facebook/mask2former-resnet50-cityscapes-semantic',
    ('cityscapes', 'semantic', 'resnet-101')     : 'facebook/mask2former-resnet101-cityscapes-semantic',
    ('cityscapes', 'semantic', 'swin-t')         : 'facebook/mask2former-swin-tiny-cityscapes-semantic',
    ('cityscapes', 'semantic', 'swin-s')         : 'facebook/mask2former-swin-small-cityscapes-semantic',
    ('cityscapes', 'semantic', 'swin-b (in21k)') : 'facebook/mask2former-swin-base-in21k-cityscapes-semantic',
    ('cityscapes', 'semantic', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-cityscapes-semantic',
    ('cityscapes', 'panoptic', 'resnet-50')      : 'facebook/mask2former-resnet50-cityscapes-panoptic',
    ('cityscapes', 'panoptic', 'resnet-101')     : 'facebook/mask2former-resnet101-cityscapes-panoptic',
    ('cityscapes', 'panoptic', 'swin-t')         : 'facebook/mask2former-swin-tiny-cityscapes-panoptic',
    ('cityscapes', 'panoptic', 'swin-s')         : 'facebook/mask2former-swin-small-cityscapes-panoptic',
    ('cityscapes', 'panoptic', 'swin-b (in21k)') : 'facebook/mask2former-swin-base-in21k-cityscapes-panoptic',
    ('cityscapes', 'panoptic', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-cityscapes-panoptic',
    ('cityscapes', 'instance', 'resnet-50')      : 'facebook/mask2former-resnet50-cityscapes-instance',
    ('cityscapes', 'instance', 'resnet-101')     : 'facebook/mask2former-resnet101-cityscapes-instance',
    ('cityscapes', 'instance', 'swin-t')         : 'facebook/mask2former-swin-tiny-cityscapes-instance',
    ('cityscapes', 'instance', 'swin-s')         : 'facebook/mask2former-swin-small-cityscapes-instance',
    ('cityscapes', 'instance', 'swin-b (in21k)') : 'facebook/mask2former-swin-base-in21k-cityscapes-instance',
    ('cityscapes', 'instance', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-cityscapes-instance',
    # ── COCO ────────────────────────────────────────────────────────────────
    ('coco', 'panoptic', 'resnet-50')      : 'facebook/mask2former-resnet50-coco-panoptic',
    ('coco', 'panoptic', 'resnet-101')     : 'facebook/mask2former-resnet101-coco-panoptic',
    ('coco', 'panoptic', 'swin-t')         : 'facebook/mask2former-swin-tiny-coco-panoptic',
    ('coco', 'panoptic', 'swin-s')         : 'facebook/mask2former-swin-small-coco-panoptic',
    ('coco', 'panoptic', 'swin-b')         : 'facebook/mask2former-swin-base-coco-panoptic',
    ('coco', 'panoptic', 'swin-b (in21k)') : 'facebook/mask2former-swin-base-in21k-coco-panoptic',
    ('coco', 'panoptic', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-coco-panoptic',
    ('coco', 'instance', 'resnet-50')      : 'facebook/mask2former-resnet50-coco-instance',
    ('coco', 'instance', 'resnet-101')     : 'facebook/mask2former-resnet101-coco-instance',
    ('coco', 'instance', 'swin-t')         : 'facebook/mask2former-swin-tiny-coco-instance',
    ('coco', 'instance', 'swin-s')         : 'facebook/mask2former-swin-small-coco-instance',
    ('coco', 'instance', 'swin-b')         : 'facebook/mask2former-swin-base-coco-instance',
    ('coco', 'instance', 'swin-b (in21k)') : 'facebook/mask2former-swin-base-in21k-coco-instance',
    ('coco', 'instance', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-coco-instance',
    # ── Mapillary Vistas ────────────────────────────────────────────────────
    ('mapillary vistas', 'semantic', 'resnet-50')      : 'facebook/mask2former-resnet50-mapillary-vistas-semantic',
    ('mapillary vistas', 'semantic', 'swin-t')         : 'facebook/mask2former-swin-tiny-mapillary-vistas-semantic',
    ('mapillary vistas', 'semantic', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-mapillary-vistas-semantic',
    ('mapillary vistas', 'panoptic', 'resnet-50')      : 'facebook/mask2former-resnet50-mapillary-vistas-panoptic',
    ('mapillary vistas', 'panoptic', 'swin-l (in21k)') : 'facebook/mask2former-swin-large-mapillary-vistas-panoptic',
}

# ─── Resolve checkpoint ─────────────────────────────────────────────────────
_key = (DATASET.lower(), TASK.lower(), BACKBONE.lower())
MODEL_ID = _REGISTRY.get(_key)

if MODEL_ID is None:
    _available = [f'  • {d.title()} / {t} / {b.upper()}'
                  for (d, t, b) in _REGISTRY if d == DATASET.lower() and t == TASK.lower()]
    _hint = '\n'.join(_available) if _available else '  (no models for this dataset+task)'
    raise ValueError(
        f'\n❌  No pretrained checkpoint for: {DATASET} | {TASK} | {BACKBONE}\n'
        f'   Available backbones for {DATASET}/{TASK}:\n{_hint}\n'
        f'   → Change DATASET, TASK, or BACKBONE above.'
    )

print(f'✅ Selected checkpoint : {MODEL_ID}')
print(f'   Dataset : {DATASET}  |  Task : {TASK}  |  Backbone : {BACKBONE}')


####Cell 4 — Configure I/O paths

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 ── 📂  I/O PATHS
# ═══════════════════════════════════════════════════════════════════════

# @markdown ### Input — images to segment
# @markdown Folder on Google Drive that contains your images (jpg/png/jpeg/webp/bmp).
INPUT_FOLDER = '/content'  # @param {type: 'string'}

# @markdown ### Output — where results are saved
OUTPUT_FOLDER = '/content/'  # @param {type: 'string'}

# @markdown ### Save outputs to Google Drive?
SAVE_TO_DRIVE = True  # @param {type: 'boolean'}

# ─── Validate / create folders ───────────────────────────────────────────────
import os
from pathlib import Path

in_root  = Path(INPUT_FOLDER)
out_root = Path(OUTPUT_FOLDER)

if not in_root.exists():
    raise FileNotFoundError(f'❌ Input folder not found: {in_root}\n   Create it on Drive and add images.')

EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
IMAGE_PATHS = sorted([p for p in in_root.rglob('*') if p.suffix.lower() in EXTS])

if not IMAGE_PATHS:
    raise FileNotFoundError(f'❌ No images found in {in_root}  (looked for {EXTS})')

if SAVE_TO_DRIVE:
    for sub in ['segmented', 'overlay', 'panels']:
        (out_root / sub).mkdir(parents=True, exist_ok=True)

print(f'✅ Found {len(IMAGE_PATHS)} image(s) in {in_root}')
print(f'   Outputs → {out_root}  (save={SAVE_TO_DRIVE})')


####Cell 5 — Set inference & visualisation options

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 ── ⚙️  INFERENCE & VISUALISATION OPTIONS
# ═══════════════════════════════════════════════════════════════════════

# @markdown ### 🎨 Overlay alpha (0 = original, 1 = full mask)
OVERLAY_ALPHA = 0.5  # @param {type: 'slider', min: 0.1, max: 0.9, step: 0.05}

# @markdown ### 🔢 Instance / panoptic confidence threshold
CONFIDENCE_THRESHOLD = 0.65  # @param {type: 'slider', min: 0.1, max: 0.95, step: 0.05}

# @markdown ### 📐 Max image size (pixels, longer edge). Reduces GPU memory.
MAX_SIZE = 1024  # @param [512, 768, 1024, 1280, 1920] {type: 'raw'}

# @markdown ---
# @markdown ### 💾 Output formats to save
SAVE_SEGMAP  = True   # @param {type: 'boolean'}
SAVE_OVERLAY = True   # @param {type: 'boolean'}
SAVE_PANEL   = True   # @param {type: 'boolean'}
SAVE_STATS   = True   # @param {type: 'boolean'}

# @markdown ### 🖼️ Panel DPI (higher = larger file)
PANEL_DPI = 150  # @param [72, 100, 150, 200, 300] {type: 'raw'}

print('⚙️  Inference options set.')
print(f'   Overlay alpha       : {OVERLAY_ALPHA}')
print(f'   Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'   Max image size      : {MAX_SIZE} px')
print(f'   Save segmap/overlay/panel/stats: {SAVE_SEGMAP}/{SAVE_OVERLAY}/{SAVE_PANEL}/{SAVE_STATS}')


####Cell 6 — Load model (auto-resolves HuggingFace checkpoint)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 ── 🤖  LOAD MODEL   (first run downloads weights, ~300–900 MB)
# ═══════════════════════════════════════════════════════════════════════
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches
import warnings, random, colorsys
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')

print(f'⬇️   Loading  {MODEL_ID} …')
processor = Mask2FormerImageProcessor.from_pretrained(MODEL_ID)
model     = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_ID).to(DEVICE).eval()

# ─── Class labels from model config ─────────────────────────────────────────
id2label    = model.config.id2label          # {int: str}
CLASS_NAMES = [id2label[i] for i in range(len(id2label))]
NUM_CLASSES = len(CLASS_NAMES)

# ════════════════════════════════════════════════════════════════════════════
# MANUAL COLOUR PALETTES  (index = class ID)
# ════════════════════════════════════════════════════════════════════════════

# ── ADE20K  (150 classes) — exact RGB from official colour table ─────────────
_ADE20K_NAMES = [
    'wall',
    'building;edifice',
    'sky',
    'floor;flooring',
    'tree',
    'ceiling',
    'road;route',
    'bed',
    'windowpane;window',
    'grass',
    'cabinet',
    'sidewalk;pavement',
    'person;individual;someone;somebody;mortal;soul',
    'earth;ground',
    'door;double;door',
    'table',
    'mountain;mount',
    'plant;flora;plant;life',
    'curtain;drape;drapery;mantle;pall',
    'chair',
    'car;auto;automobile;machine;motorcar',
    'water',
    'painting;picture',
    'sofa;couch;lounge',
    'shelf',
    'house',
    'sea',
    'mirror',
    'rug;carpet;carpeting',
    'field',
    'armchair',
    'seat',
    'fence;fencing',
    'desk',
    'rock;stone',
    'wardrobe;closet;press',
    'lamp',
    'bathtub;bathing;tub;bath;tub',
    'railing;rail',
    'cushion',
    'base;pedestal;stand',
    'box',
    'column;pillar',
    'signboard;sign',
    'chest;of;drawers;chest;bureau;dresser',
    'counter',
    'sand',
    'sink',
    'skyscraper',
    'fireplace;hearth;open;fireplace',
    'refrigerator;icebox',
    'grandstand;covered;stand',
    'path',
    'stairs;steps',
    'runway',
    'case;display;case;showcase;vitrine',
    'pool;table;billiard;table;snooker;table',
    'pillow',
    'screen;door;screen',
    'stairway;staircase',
    'river',
    'bridge;span',
    'bookcase',
    'blind;screen',
    'coffee;table;cocktail;table',
    'toilet;can;commode;crapper;pot;potty;stool;throne',
    'flower',
    'book',
    'hill',
    'bench',
    'countertop',
    'stove;kitchen;stove;range;kitchen;range;cooking;stove',
    'palm;palm;tree',
    'kitchen;island',
    'computer;computing;machine;computing;device;data;processor;electronic;computer;information;processing;system',
    'swivel;chair',
    'boat',
    'bar',
    'arcade;machine',
    'hovel;hut;hutch;shack;shanty',
    'bus;autobus;coach;charabanc;double-decker;jitney;motorbus;motorcoach;omnibus;passenger;vehicle',
    'towel',
    'light;light;source',
    'truck;motortruck',
    'tower',
    'chandelier;pendant;pendent',
    'awning;sunshade;sunblind',
    'streetlight;street;lamp',
    'booth;cubicle;stall;kiosk',
    'television;television;receiver;television;set;tv;tv;set;idiot;box;boob;tube;telly;goggle;box',
    'airplane;aeroplane;plane',
    'dirt;track',
    'apparel;wearing;apparel;dress;clothes',
    'pole',
    'land;ground;soil',
    'bannister;banister;balustrade;balusters;handrail',
    'escalator;moving;staircase;moving;stairway',
    'ottoman;pouf;pouffe;puff;hassock',
    'bottle',
    'buffet;counter;sideboard',
    'poster;posting;placard;notice;bill;card',
    'stage',
    'van',
    'ship',
    'fountain',
    'conveyer;belt;conveyor;belt;conveyer;conveyor;transporter',
    'canopy',
    'washer;automatic;washer;washing;machine',
    'plaything;toy',
    'swimming;pool;swimming;bath;natatorium',
    'stool',
    'barrel;cask',
    'basket;handbasket',
    'waterfall;falls',
    'tent;collapsible;shelter',
    'bag',
    'minibike;motorbike',
    'cradle',
    'oven',
    'ball',
    'food;solid;food',
    'step;stair',
    'tank;storage;tank',
    'trade;name;brand;name;brand;marque',
    'microwave;microwave;oven',
    'pot;flowerpot',
    'animal;animate;being;beast;brute;creature;fauna',
    'bicycle;bike;wheel;cycle',
    'lake',
    'dishwasher;dish;washer;dishwashing;machine',
    'screen;silver;screen;projection;screen',
    'blanket;cover',
    'sculpture',
    'hood;exhaust;hood',
    'sconce',
    'vase',
    'traffic;light;traffic;signal;stoplight',
    'tray',
    'ashcan;trash;can;garbage;can;wastebin;ash;bin;ash-bin;ashbin;dustbin;trash;barrel;trash;bin',
    'fan',
    'pier;wharf;wharfage;dock',
    'crt;screen',
    'plate',
    'monitor;monitoring;device',
    'bulletin;board;notice;board',
    'shower',
    'radiator',
    'glass;drinking;glass',
    'clock',
    'flag',
]

_ADE20K_PALETTE = [
    [120, 120, 120],
    [180, 120, 120],
    [6, 230, 230],
    [80, 50, 50],
    [4, 200, 3],
    [120, 120, 80],
    [140, 140, 140],
    [204, 5, 255],
    [230, 230, 230],
    [4, 250, 7],
    [224, 5, 255],
    [235, 255, 7],
    [150, 5, 61],
    [120, 120, 70],
    [8, 255, 51],
    [255, 6, 82],
    [143, 255, 140],
    [204, 255, 4],
    [255, 51, 7],
    [204, 70, 3],
    [0, 102, 200],
    [61, 230, 250],
    [255, 6, 51],
    [11, 102, 255],
    [255, 7, 71],
    [255, 9, 224],
    [9, 7, 230],
    [220, 220, 220],
    [255, 9, 92],
    [112, 9, 255],
    [8, 255, 214],
    [7, 255, 224],
    [255, 184, 6],
    [10, 255, 71],
    [255, 41, 10],
    [7, 255, 255],
    [224, 255, 8],
    [102, 8, 255],
    [255, 61, 6],
    [255, 194, 7],
    [255, 122, 8],
    [0, 255, 20],
    [255, 8, 41],
    [255, 5, 153],
    [6, 51, 255],
    [235, 12, 255],
    [160, 150, 20],
    [0, 163, 255],
    [140, 140, 140],
    [250, 10, 15],
    [20, 255, 0],
    [31, 255, 0],
    [255, 31, 0],
    [255, 224, 0],
    [153, 255, 0],
    [0, 0, 255],
    [255, 71, 0],
    [0, 235, 255],
    [0, 173, 255],
    [31, 0, 255],
    [11, 200, 200],
    [255, 82, 0],
    [0, 255, 245],
    [0, 61, 255],
    [0, 255, 112],
    [0, 255, 133],
    [255, 0, 0],
    [255, 163, 0],
    [255, 102, 0],
    [194, 255, 0],
    [0, 143, 255],
    [51, 255, 0],
    [0, 82, 255],
    [0, 255, 41],
    [0, 255, 173],
    [10, 0, 255],
    [173, 255, 0],
    [0, 255, 153],
    [255, 92, 0],
    [255, 0, 255],
    [255, 0, 245],
    [255, 0, 102],
    [255, 173, 0],
    [255, 0, 20],
    [255, 184, 184],
    [0, 31, 255],
    [0, 255, 61],
    [0, 71, 255],
    [255, 0, 204],
    [0, 255, 194],
    [0, 255, 82],
    [0, 10, 255],
    [0, 112, 255],
    [51, 0, 255],
    [0, 194, 255],
    [0, 122, 255],
    [0, 255, 163],
    [255, 153, 0],
    [0, 255, 10],
    [255, 112, 0],
    [143, 255, 0],
    [82, 0, 255],
    [163, 255, 0],
    [255, 235, 0],
    [8, 184, 170],
    [133, 0, 255],
    [0, 255, 92],
    [184, 0, 255],
    [255, 0, 31],
    [0, 184, 255],
    [0, 214, 255],
    [255, 0, 112],
    [92, 255, 0],
    [0, 224, 255],
    [112, 224, 255],
    [70, 184, 160],
    [163, 0, 255],
    [153, 0, 255],
    [71, 255, 0],
    [255, 0, 163],
    [255, 204, 0],
    [255, 0, 143],
    [0, 255, 235],
    [133, 255, 0],
    [255, 0, 235],
    [245, 0, 255],
    [255, 0, 122],
    [255, 245, 0],
    [10, 190, 212],
    [214, 255, 0],
    [0, 204, 255],
    [20, 0, 255],
    [255, 255, 0],
    [0, 153, 255],
    [0, 41, 255],
    [0, 255, 204],
    [41, 0, 255],
    [41, 255, 0],
    [173, 0, 255],
    [0, 245, 255],
    [71, 0, 255],
    [122, 0, 255],
    [0, 255, 184],
    [0, 92, 255],
    [184, 255, 0],
    [0, 133, 255],
    [255, 214, 0]
]

# ── Cityscapes  (19 classes) ────────────────────────────────────────────────
_CITYSCAPES_NAMES = [
    'road', 'sidewalk', 'building', 'wall', 'fence',
    'pole', 'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck',
    'bus', 'train', 'motorcycle', 'bicycle',
]

_CITYSCAPES_PALETTE = [
    [128, 64,128],   #  0: road
    [244, 35,232],   #  1: sidewalk
    [ 70, 70, 70],   #  2: building
    [102,102,156],   #  3: wall
    [190,153,153],   #  4: fence
    [153,153,153],   #  5: pole
    [250,170, 30],   #  6: traffic light
    [220,220,  0],   #  7: traffic sign
    [  0,255,  0],   #  8: vegetation
    [152,251,152],   #  9: terrain
    [ 70,130,180],   # 10: sky
    [220, 20, 60],   # 11: person
    [255,  0,  0],   # 12: rider
    [  0,  0,142],   # 13: car
    [  0,  0, 70],   # 14: truck
    [  0, 60,100],   # 15: bus
    [  0, 80,100],   # 16: train
    [  0,  0,230],   # 17: motorcycle
    [119, 11, 32],   # 18: bicycle
]

# ── Auto-generate for COCO / Mapillary Vistas ────────────────────────────────
def _auto_palette(n):
    import colorsys
    return [
        [int(r*255), int(g*255), int(b*255)]
        for r, g, b in (
            colorsys.hsv_to_rgb((i / max(n, 1)) % 1.0, 0.75, 0.90)
            for i in range(n)
        )
    ]

# ── Select palette & names based on chosen DATASET ───────────────────────────
_DATASET_KEY = DATASET.lower()

if _DATASET_KEY == 'ade20k':
    COLOR_PALETTE = _ADE20K_PALETTE
    if len(_ADE20K_NAMES) == NUM_CLASSES:
        CLASS_NAMES = _ADE20K_NAMES

elif _DATASET_KEY == 'cityscapes':
    COLOR_PALETTE = _CITYSCAPES_PALETTE
    if len(_CITYSCAPES_NAMES) == NUM_CLASSES:
        CLASS_NAMES = _CITYSCAPES_NAMES

else:  # COCO, Mapillary Vistas
    COLOR_PALETTE = _auto_palette(NUM_CLASSES)

# Safety: pad or truncate to exactly NUM_CLASSES
while len(COLOR_PALETTE) < NUM_CLASSES:
    COLOR_PALETTE.append(_auto_palette(1)[0])
COLOR_PALETTE = COLOR_PALETTE[:NUM_CLASSES]

print(f'✅ Model ready  —  {NUM_CLASSES} classes  |  task: {TASK}  |  dataset: {DATASET}')
print(f'   Palette: {"manual (" + DATASET + ")" if _DATASET_KEY in ("ade20k","cityscapes") else "auto-generated"}')
print(f'   First 8 classes: {CLASS_NAMES[:8]}')


####Cell 7 — Run batch inference

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 ── 🔧  HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════
import torchvision.transforms.functional as TF

# ── Image loader with optional resize ────────────────────────────────────────
def load_image(path: Path) -> Image.Image:
    img = Image.open(path).convert('RGB')
    w, h = img.size
    scale = MAX_SIZE / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
    return img


# ── Semantic: returns (label_map, seg_image) ─────────────────────────────────
def infer_semantic(image: Image.Image):
    inputs = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    result  = processor.post_process_semantic_segmentation(
        outputs, target_sizes=[image.size[::-1]])[0]  # (H, W)
    lbl_map = result.cpu().numpy().astype(np.int32)
    H, W    = lbl_map.shape
    seg_rgb = np.zeros((H, W, 3), dtype=np.uint8)
    for cls_id, color in enumerate(COLOR_PALETTE[:NUM_CLASSES]):
        seg_rgb[lbl_map == cls_id] = color
    return lbl_map, Image.fromarray(seg_rgb)


# ── Panoptic: returns (id_map, seg_image, segments_info) ─────────────────────
def infer_panoptic(image: Image.Image):
    inputs = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    result = processor.post_process_panoptic_segmentation(
        outputs,
        threshold=CONFIDENCE_THRESHOLD,
        target_sizes=[image.size[::-1]])[0]
    seg_map  = result['segmentation'].cpu().numpy()          # (H,W) with segment IDs
    seg_info = result['segments_info']                        # list of dicts
    H, W     = seg_map.shape
    seg_rgb  = np.zeros((H, W, 3), dtype=np.uint8)
    rng = random.Random(42)
    for seg in seg_info:
        sid    = seg['id']
        lbl    = seg['label_id']
        color  = COLOR_PALETTE[lbl] if lbl < len(COLOR_PALETTE) else \
                 (rng.randint(80,230), rng.randint(80,230), rng.randint(80,230))
        seg_rgb[seg_map == sid] = color
    return seg_map, Image.fromarray(seg_rgb), seg_info


# ── Instance: returns (id_map, seg_image, segments_info) ─────────────────────
def infer_instance(image: Image.Image):
    inputs = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    result = processor.post_process_instance_segmentation(
        outputs,
        threshold=CONFIDENCE_THRESHOLD,
        target_sizes=[image.size[::-1]])[0]
    seg_map  = result['segmentation'].cpu().numpy()
    seg_info = result['segments_info']
    H, W     = seg_map.shape
    seg_rgb  = np.zeros((H, W, 3), dtype=np.uint8)
    rng = random.Random(42)
    for seg in seg_info:
        sid   = seg['id']
        lbl   = seg['label_id']
        color = COLOR_PALETTE[lbl] if lbl < len(COLOR_PALETTE) else \
                (rng.randint(80,230), rng.randint(80,230), rng.randint(80,230))
        seg_rgb[seg_map == sid] = color
    return seg_map, Image.fromarray(seg_rgb), seg_info


# ── Unified inference dispatcher ─────────────────────────────────────────────
def run_inference(image: Image.Image):
    """Returns (lbl_map_np, seg_pil, extra_info_or_None)."""
    if TASK == 'semantic':
        lbl, seg = infer_semantic(image)
        return lbl, seg, None
    elif TASK == 'panoptic':
        return infer_panoptic(image)
    else:
        return infer_instance(image)


# ── Overlay builder ──────────────────────────────────────────────────────────
def make_overlay(original: Image.Image, seg: Image.Image) -> Image.Image:
    orig_np = np.array(original).astype(float)
    seg_np  = np.array(seg).astype(float)
    blend   = (orig_np * (1 - OVERLAY_ALPHA) + seg_np * OVERLAY_ALPHA).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blend)


# ── 3-panel figure (original | seg | overlay) ────────────────────────────────
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch

def make_panel(original, seg, overlay, title='', seg_info=None):

    seg_array = np.array(seg)

    # ── Compute class coverage (semantic only) ────────────────────────────────
    sorted_classes, sorted_counts = [], []
    if TASK == 'semantic':
        if seg_array.ndim == 3:
            palette  = np.array(COLOR_PALETTE[:NUM_CLASSES], dtype=np.int16)
            flat     = seg_array.reshape(-1, 3).astype(np.int16)
            dists    = np.sum(np.abs(flat[:, None, :] - palette[None, :, :]), axis=2)
            flat_idx = np.argmin(dists, axis=1).astype(np.int32)
            flat_idx[dists[np.arange(len(flat)), flat_idx] > 10] = -1
        else:
            flat_idx = seg_array.flatten().astype(np.int32)

        unique, counts = np.unique(flat_idx[flat_idx >= 0], return_counts=True)
        order          = np.argsort(-counts)
        sorted_classes = unique[order].tolist()
        sorted_counts  = counts[order].tolist()

    # ── Layout via GridSpec ───────────────────────────────────────────────────
    max_legend = 20
    n          = min(len(sorted_classes), max_legend)
    ncol_leg   = 4
    nrow_leg   = math.ceil(n / ncol_leg) if n > 0 else 0

    # Legend row height in inches; image row fixed at 5 in
    img_h    = 5.0
    leg_h    = nrow_leg * 0.38          # ~0.38 in per legend row
    fig_h    = img_h + leg_h + 0.5      # 0.5 in for title breathing room

    fig = plt.figure(figsize=(18, fig_h))

    if n > 0:
        gs = gridspec.GridSpec(
            2, 3,
            figure=fig,
            height_ratios=[img_h, leg_h],
            hspace=0.15,              # tight but not squished
            wspace=0.04,
        )
        axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
        legend_ax = fig.add_subplot(gs[1, :])   # full-width bottom row
    else:
        gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.04)
        axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
        legend_ax = None

    # ── Draw the three image panels ───────────────────────────────────────────
    labels = ['Original', f'{TASK.title()} Segmentation', 'Overlay']
    for ax, im, lbl in zip(axes, [original, seg, overlay], labels):
        ax.imshow(im)
        ax.set_title(lbl, fontsize=13, fontweight='bold', pad=6)
        ax.axis('off')

    if TASK != 'semantic' and seg_info is not None:
        axes[1].set_title(
            f'{TASK.title()} — {len(seg_info)} segments detected',
            fontsize=13, fontweight='bold', pad=6
        )

    # ── Draw legend in the dedicated bottom axes ──────────────────────────────
    if legend_ax is not None:
        legend_ax.set_axis_off()
        total_px = sum(sorted_counts)

        col_w = 1.0 / ncol_leg          # fraction of axes width per column

        for rank in range(n):
            cls_id   = sorted_classes[rank]
            px_count = sorted_counts[rank]
            col      = rank % ncol_leg
            row      = rank // ncol_leg

            # x: left edge of this column; y: top-down rows
            x0  = col * col_w + 0.01
            # normalise row so row 0 is near top of the legend axes
            y0  = 1.0 - (row + 1) / nrow_leg + 0.05

            pct = 100 * px_count / total_px
            rgb = np.array(COLOR_PALETTE[cls_id]) / 255

            # Color swatch
            legend_ax.add_patch(FancyBboxPatch(
                (x0, y0), 0.016, 0.55 / nrow_leg,
                boxstyle='round,pad=0.002',
                facecolor=rgb, edgecolor='white', linewidth=0.5,
                transform=legend_ax.transAxes, clip_on=False
            ))

            # ID badge
            badge_x = x0 + 0.019
            legend_ax.add_patch(FancyBboxPatch(
                (badge_x, y0), 0.020, 0.55 / nrow_leg,
                boxstyle='round,pad=0.002',
                facecolor='#222222', edgecolor='none',
                transform=legend_ax.transAxes, clip_on=False
            ))
            legend_ax.text(
                badge_x + 0.010, y0 + 0.28 / nrow_leg,
                f'#{cls_id}',
                transform=legend_ax.transAxes,
                fontsize=6.5, fontweight='bold',
                color='white', ha='center', va='center'
            )

            # Class name + %
            legend_ax.text(
                x0 + 0.042, y0 + 0.28 / nrow_leg,
                f'{CLASS_NAMES[cls_id]}  {pct:.1f}%',
                transform=legend_ax.transAxes,
                fontsize=8, va='center', color='#222222'
            )

    if title:
        fig.suptitle(title, fontsize=11, y=1.0)

    return fig


# ── Pixel statistics (per class) ─────────────────────────────────────────────
def pixel_stats_semantic(lbl_map: np.ndarray, filename: str) -> dict:
    total = lbl_map.size
    row   = {'filename': filename, 'status': 'ok',
             'width': lbl_map.shape[1], 'height': lbl_map.shape[0],
             'total_pixels': total, 'task': TASK, 'model': MODEL_ID}
    for i, name in enumerate(CLASS_NAMES):
        cnt = int((lbl_map == i).sum())
        row[f'{name}_pixels']  = cnt
        row[f'{name}_percent'] = round(cnt / total * 100, 4)
    return row


def pixel_stats_panoptic_instance(seg_map: np.ndarray, seg_info, filename: str) -> dict:
    total = seg_map.size
    row   = {'filename': filename, 'status': 'ok',
             'width': seg_map.shape[1], 'height': seg_map.shape[0],
             'total_pixels': total, 'num_segments': len(seg_info),
             'task': TASK, 'model': MODEL_ID}
    # Aggregate by semantic class
    cls_pixels = {}
    for seg in seg_info:
        lbl   = seg['label_id']
        name  = id2label.get(lbl, f'class_{lbl}')
        cnt   = int((seg_map == seg['id']).sum())
        cls_pixels[name] = cls_pixels.get(name, 0) + cnt
    for name, cnt in cls_pixels.items():
        row[f'{name}_pixels']  = cnt
        row[f'{name}_percent'] = round(cnt / total * 100, 4)
    return row


print('✅ Helper functions defined.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  BATCH INFERENCE LOOP
# ═══════════════════════════════════════════════════════════════════════
records = []
failed  = []

for img_path in tqdm(IMAGE_PATHS, desc='Segmenting images'):
    stem = img_path.stem
    try:
        original = load_image(img_path)

        # ── Inference ──────────────────────────────────────────────────────
        lbl_map, seg_img, extra = run_inference(original)

        # ── Derived images ─────────────────────────────────────────────────
        overlay_img = make_overlay(original, seg_img)

        # ── Pixel statistics ───────────────────────────────────────────────
        if TASK == 'semantic':
            rec = pixel_stats_semantic(lbl_map, img_path.name)
        else:
            rec = pixel_stats_panoptic_instance(lbl_map, extra, img_path.name)
        records.append(rec)

        # ── Save outputs ───────────────────────────────────────────────────
        if SAVE_TO_DRIVE:
            if SAVE_SEGMAP:
                seg_img.save(out_root / 'segmented' / f'{stem}_seg.png')
            if SAVE_OVERLAY:
                overlay_img.save(out_root / 'overlay' / f'{stem}_overlay.png')
            if SAVE_PANEL:
                fig = make_panel(original, seg_img, overlay_img,
                                 title=f'{img_path.name}  [{MODEL_ID}]',
                                 seg_info=extra)
                fig.savefig(out_root / 'panels' / f'{stem}_panel.png',
                            dpi=PANEL_DPI, bbox_inches='tight')
                plt.close(fig)

    except Exception as e:
        failed.append({'filename': img_path.name, 'status': 'error', 'error': str(e)})
        tqdm.write(f'  ⚠️  {img_path.name}: {e}')

df = pd.DataFrame(records + failed)
ok = (df['status'] == 'ok').sum() if 'status' in df.columns else 0
print(f'\n✅ Done: {ok}/{len(IMAGE_PATHS)} images processed successfully.')
if failed:
    print(f'   ⚠️  {len(failed)} failed — check the error column in df.')


####Cell 8 — Preview results

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 ── 🖼️  QUICK PREVIEW  (first 3 results inline)
# ═══════════════════════════════════════════════════════════════════════
N_PREVIEW = 3  # @param {type: 'integer'}

for img_path in IMAGE_PATHS[:N_PREVIEW]:
    try:
        original = load_image(img_path)
        lbl_map, seg_img, extra = run_inference(original)
        overlay_img = make_overlay(original, seg_img)
        fig = make_panel(original, seg_img, overlay_img,
                         title=img_path.name, seg_info=extra)
        plt.show()
        plt.close(fig)
    except Exception as e:
        print(f'⚠️  Could not preview {img_path.name}: {e}')


####Cell 9 — Export pixel stats (CSV / XLSX)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📊  EXPORT PIXEL STATISTICS
# ═══════════════════════════════════════════════════════════════════════
if not SAVE_STATS or not SAVE_TO_DRIVE:
    print('ℹ️  Stats export skipped (SAVE_STATS or SAVE_TO_DRIVE is False).')
else:
    # ── CSV ────────────────────────────────────────────────────────────────
    csv_path = out_root / 'pixel_counts.csv'
    df.to_csv(csv_path, index=False)
    print(f'✅ CSV  saved → {csv_path}')

    # ── XLSX (two sheets: per-image + summary) ─────────────────────────────
    xlsx_path = out_root / 'pixel_counts.xlsx'
    pct_cols  = [c for c in df.columns if c.endswith('_percent')]
    ok_df     = df[df['status'] == 'ok'] if 'status' in df.columns else df

    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Per-Image', index=False)
        if len(ok_df) > 0 and pct_cols:
            summary = ok_df[pct_cols].agg(['mean', 'std', 'min', 'max']).T
            summary.index = [c.replace('_percent', '') for c in summary.index]
            summary.to_excel(writer, sheet_name='Summary')
    print(f'✅ XLSX saved → {xlsx_path}')


####Cell 10 — Class distribution chart

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📈  MEAN CLASS DISTRIBUTION CHART
# ═══════════════════════════════════════════════════════════════════════
pct_cols = [c for c in df.columns if c.endswith('_percent')]
ok_df    = df[df['status'] == 'ok'] if 'status' in df.columns else df

if len(ok_df) == 0 or not pct_cols:
    print('ℹ️  No data to chart.')
else:
    means  = ok_df[pct_cols].mean().sort_values(ascending=False)
    labels = [c.replace('_percent', '') for c in means.index]

    def _lbl_color(name):
        if name in CLASS_NAMES:
            return np.array(COLOR_PALETTE[CLASS_NAMES.index(name)]) / 255
        return (0.55, 0.55, 0.55)

    colors = [_lbl_color(l) for l in labels]

    fig, ax = plt.subplots(figsize=(max(14, len(labels)*0.55), 5))
    bars = ax.bar(labels, means.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_ylabel('Mean pixel %', fontsize=12)
    ax.set_title(
        f'Mean class distribution  |  {DATASET} / {TASK} / {BACKBONE}\n'
        f'Model: {MODEL_ID}  |  n={len(ok_df)} images',
        fontsize=12, fontweight='bold')
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    for bar, val in zip(bars, means.values):
        if val > 0.5:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    f'{val:.1f}%', ha='center', va='bottom', fontsize=7)
    plt.tight_layout()
    plt.show()

    if SAVE_TO_DRIVE and SAVE_STATS:
        chart_path = out_root / 'class_distribution.png'
        fig.savefig(chart_path, dpi=150, bbox_inches='tight')
        print(f'✅ Chart saved → {chart_path}')
    plt.close(fig)


## 📂 Output folder structure

```
<OUTPUT_FOLDER>/
├── segmented/              ← colour label maps              (*_seg.png)
├── overlay/                ← original blended with mask     (*_overlay.png)
├── panels/                 ← 3-panel figure per image       (*_panel.png)
├── pixel_counts.csv        ← per-image pixel counts (flat CSV)
├── pixel_counts.xlsx       ← same + summary sheet
└── class_distribution.png  ← mean class distribution chart
```

**Spreadsheet columns:**  
`filename · status · width · height · total_pixels · task · model · {class}_pixels · {class}_percent`

---

## 🗺️ Full Model Matrix

| Dataset | Semantic | Panoptic | Instance |
|---------|----------|----------|----------|
| **ADE20K** | R50, R101, Swin-T/S/B/B†/L† | R50, Swin-L† | R50, Swin-L† |
| **Cityscapes** | R50, R101, Swin-T/S/B†/L† | R50, R101, Swin-T/S/B†/L† | R50, R101, Swin-T/S/B†/L† |
| **COCO** | *(not available)* | R50, R101, Swin-T/S/B/B†/L† | R50, R101, Swin-T/S/B/B†/L† |
| **Mapillary Vistas** | R50, Swin-T, Swin-L† | R50, Swin-L† | *(not available)* |

> † = pretrained on ImageNet-21k
